## scenario analysis with Brightway

how we can modify values of the matrices that underpin the LCA calculations

In [ ]:
import bw2data as bd
import bw_processing as bwp
import bw2analyzer as bwa
import bw2calc as bc
import bw2io as bi
import pandas as pd
import numpy as np

import reed


[some other tutorials](https://github.com/brightway-lca/from-the-ground-up/blob/main/basic%20tasks/Define%20a%20datapackage%20interface.ipynb)

[papers](https://onlinelibrary.wiley.com/doi/pdf/10.1111/jiec.13510)

In [2]:
bd.projects.set_current('reed')

In [3]:
bd.databases

Databases dictionary with 5 object(s):
	Mobility example
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

In [4]:
from reed import utils

In [5]:
CONFIG = utils.Config()

## import REED inventories 
(only fi neeed)

In [78]:
inventory_path = CONFIG.REED_PATH/"Model"/"REED_model_20260203.xlsx"

In [79]:
CONFIG.REED_PATH

WindowsPath('C:/Users/MiguelFernandezAstud/OneDrive - 2.-0 LCA Consultants ApS/Intranet - HorizonEurope/EU_REED_1277')

In [80]:
CONFIG.REED_PATH.exists()

True

In [88]:
importer = bi.importers.ExcelImporter(inventory_path)

Extracted 3 worksheets in 0.28 seconds


In [89]:
importer.apply_strategies()

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.20 seconds


In [90]:
importer.match_database(fields=['name','unit','location'])

Applying strategy: link_iterable_by_fields


In [91]:
importer.match_database('bonsai_231231',fields=['name','unit','location','activity type'])

Applying strategy: link_iterable_by_fields


In [92]:
importer.match_database(bd.config.biosphere,fields=['name','unit','categories'])

Applying strategy: link_iterable_by_fields


In [93]:
assert importer.all_linked

Graph statistics for `reed` importer:
30 graph nodes:
	None: 30
173 graph edges:
	technosphere: 141
	production: 30
	biosphere: 2
173 edges to the following databases:
	bonsai_231231: 89
	reed: 82
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):




In [94]:
importer.write_database()

100%|██████████| 30/30 [00:00<00:00, 1216.73it/s]


10:09:34+0100 [info     ] Vacuuming database            
Created database: reed


## standard LCA calculation 

In [95]:
reed_db = bd.Database('reed')

In [17]:
wwr_case = reed_db.get(name="WWR piece")

In [19]:
for f in wwr_case.exchanges():
    print(f)

Exchange: 1 unit 'WWR piece' (unit, DE, None) to 'WWR piece' (unit, DE, None)>
Exchange: 12450 kilogram 'steel production' (kilogram, CN, None) to 'WWR piece' (unit, DE, None)>
Exchange: 825 kilogram 'milling' (kilogram, GB, None) to 'WWR piece' (unit, DE, None)>
Exchange: 1 hour 'bending' (hour, DE, None) to 'WWR piece' (unit, DE, None)>
Exchange: 3 hour 'sawing' (hour, DE, None) to 'WWR piece' (unit, DE, None)>
Exchange: 9.132420091324201e-05 unit 'soraluce machine, WWR' (unit, DE, None) to 'WWR piece' (unit, DE, None)>
Exchange: 0 hour 'machining, generic, hour' (hour, GB, None) to 'WWR piece' (unit, DE, None)>


bending => sawing => machining

In [20]:
steel = reed_db.get(name='steel production')
steel

'steel production' (kilogram, CN, None)

In [100]:
bending = reed_db.get(name='bending')

sawing = reed_db.get(name='sawing')

milling = reed_db.get(name='milling')

machinig = reed_db.get(name='machining, generic',unit='hour')

In [23]:
CONFIG.lcia_gwp100

('IPCC 2021', 'climate change', 'global warming potential (GWP100)')

In [24]:
lca = wwr_case.lca(method=CONFIG.lcia_gwp100)

In [25]:
lca.score

47171.25638743187

In [26]:
bwa.print_recursive_calculation(wwr_case,CONFIG.lcia_gwp100,max_level=1,cutoff=0.001)

Fraction of score | Absolute score | Amount | Activity
0001 | 4.717e+04 |     1 | 'WWR piece' (unit, DE, None)
  0.976 | 4.606e+04 | 1.245e+04 | 'steel production' (kilogram, CN, None)
  0.0204 | 960.3 |   825 | 'milling' (kilogram, GB, None)
  0.00199 | 93.94 |     3 | 'sawing' (hour, DE, None)


most of the global warming emissions come from steel production

## refresher on standard LCA algebra

From intput output 
$$C \cdot B \cdot (I- \hat Z)^{-1} y $$

LCA terms

$$LCA score = C \cdot B \cdot s$$

$$A \cdot s = f$$

- f: final demand vector
- s : scaling vector
- A : technology matrix 
- B : intervention matrix
- C : matrix of characterization factors

In [30]:
#reed_db.graph_technosphere()

In [32]:
lca.score

47171.25638743187

In [31]:
sum(lca.characterization_matrix @ lca.biosphere_matrix @ lca.supply_array)

47171.25638743181

the lca.product_dict provides a mapping between the ids of each product, and the position in the matrix

In [39]:
lca.technosphere_matrix[lca.product_dict[wwr_case.id],lca.product_dict[wwr_case.id]]

1.0

In [40]:
lca.technosphere_matrix[lca.product_dict[sawing.id],lca.activity_dict[wwr_case.id]]

-3.0

In [41]:
lca.technosphere_matrix[lca.product_dict[milling.id],lca.activity_dict[wwr_case.id]]

-825.0

In [46]:
steel_amount = abs(lca.technosphere_matrix[lca.product_dict[steel.id],lca.activity_dict[wwr_case.id]])
steel_amount

12450.0

why it is negative ? 

## bw_processing

powerful package for manipulating inventories

In [47]:
cols = pd.Index(['not yet machined','machined','very wrong'])
cols

Index(['not yet machined', 'machined', 'very wrong'], dtype='object')

In [49]:
rows = pd.MultiIndex.from_tuples([
                 (steel.id,wwr_case.id),
                 (bending.id,wwr_case.id),
                 (sawing.id,wwr_case.id),
                 (milling.id,wwr_case.id),
                 (machinig.id,wwr_case.id),
                  ],names=['product','activity'])
rows

MultiIndex([(277095282102988800, 277095282207846400),
            (277095282149126144, 277095282207846400),
            (277095282149126145, 277095282207846400),
            (277095282157514752, 277095282207846400),
            (277095282203652096, 277095282207846400)],
           names=['product', 'activity'])

imagine we get from the process process simulation hub that:
- 1) in the first scenario it took a 20% more time to bend, saw and mill the WWR piece, but the piece is not yet machined
- 2) the same as previous one but the piece was machined 40 h
- 3) an alternative to the previous one but were the piece twice as large (just for testing)

bending_hours = 1
sawing_hours = 3
milling_kg = 850
machinig_h = 0

In [103]:
values = np.array([[steel_amount,bending_hours*1.2,sawing_hours*1.2,milling_kg*1.2,machinig_h], # not yet machined
          [steel_amount,bending_hours*1.2,sawing_hours*1.2,milling_kg*1.2,40], # 40 h of machining
          [steel_amount*2,bending_hours*1.2,sawing_hours*1.2,milling_kg*1.2,40]] # imagine piece is 2 times bigger
          ).T

In [104]:
rows.shape

(5,)

In [105]:
scenarios = pd.DataFrame(values,index=rows,columns=cols)

In [107]:
scenarios

,,not yet machined,machined,very wrong
product,activity,,,
277095282102988800,277095282207846400,12450.0,12450.0,24900.0
277095282149126144,277095282207846400,1.2,1.2,1.2
277095282149126145,277095282207846400,3.6,3.6,3.6
277095282157514752,277095282207846400,1020.0,1020.0,1020.0
277095282203652096,277095282207846400,0.0,40.0,40.0


In [109]:
len(bd.Database('bonsai_231231')) + len(reed_db) == lca.technosphere_matrix.shape[0]

True

the technosphere includes all the activities in bonsai and reed, even if they are not necessarely part of the product system 

In [59]:
dp = bwp.create_datapackage(sequential=True)

In [60]:
dp.add_dynamic_array(
    matrix='technosphere_matrix',
    indices_array=scenarios.index.values.astype(bwp.INDICES_DTYPE),
    interface=scenarios.values,
    flip_array=np.full(scenarios.index.shape,True))

In [61]:
fu, data_objs , _ = bd.prepare_lca_inputs({wwr_case:1},method=CONFIG.lcia_gwp100)

In [56]:
lca = bc.LCA({wwr_case.id:1},CONFIG.lcia_gwp100)
lca.lci()
lca.lcia()
lca.score

47171.25638743187

In [57]:
lca =wwr_case.lca(CONFIG.lcia_gwp100)
lca.score


47171.25638743187

In [ ]:
lca_a = bc.LCA(demand=fu,method=CONFIG.lcia_gwp100,
               data_objs=data_objs + [dp],use_arrays=True)

In [63]:
lca_a.lci()
lca_a.lcia()
#lca_a.keep_first_iteration()

In [125]:
lca.score

47171.25638743187

In [64]:
results = []
for i in range(4):
    
    print(i)
    print(lca_a.score/lca.score)
    results.append(lca_a.score)
    steel_cons = (lca_a.technosphere_matrix[lca_a.product_dict[steel.id],lca_a.activity_dict[wwr_case.id]])
    print(steel_cons)
    next(lca_a)
    


0
1.0053359976275023
-12450.0
1
275.7638649171977
-12450.0
2
276.74032302895245
-24900.0
3
1.0053359976275023
-12450.0


after 3 iterations it goes back to the beginning of the sequence, hence we get the same results

In [70]:
next(lca_a)
lca_a.score

13054188.730339434

In [71]:
next(lca_a)
lca_a.score

47422.962099601515

In [65]:
lca = machinig.lca(method=CONFIG.lcia_gwp100)
lca.score

324017.6253074662